# Bornholm 2035 · What does the hybrid gain?
Run locally using the campaign PyPSA environment. Open this notebook inside the model repository. Copy the Slurm array's transfer directory first. Figures are annual gains in millions of 2025 euros for a common 3 GW wind build. The four strikes use signed actual-production settlement as a proxy. Report results marked for review as provisional.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'scripts/analyse_bornholm_matched.py').is_file())
sys.path.insert(0, str(repo))
from scripts.analyse_bornholm_matched import compare
DATA_DIR = None  # Set this to the transfer folder if it is outside the repo.
if DATA_DIR is None:
    batches = list((repo/'results/bornholm_2035_matched').glob('*/transfer'))
    if len(batches) != 1:
        raise ValueError("Set DATA_DIR to one complete matched trial's transfer folder.")
    DATA_DIR = batches[0]
runs, changes = compare(DATA_DIR)
display(runs.drop(columns=['source_hash','code_hash']))

## Routing and cable sizing
The first pair changes routing and any associated offshore price exposure. The second adds 200 MW on the Danish leg with its modelled investment cost. The third gives the combined effect. A pooled cost share of zero assigns bundled export legs to the landing country. One pools both legs equally, as in the original notebook. Neither resolves the unitemised common island equipment. Rent is shared equally in both. Other territories and unallocated entries remain in the reconciliation.

In [ ]:
display(changes[changes.strike == 75].round(4))
display(changes.pivot_table(index=['reference','design','pooled_cost_share'], columns='strike', values=['DK','DE','developer']).round(3))
if changes.residual.abs().max() > 0.01:
    print('Cash settlement differs from avoided system cost by more than €0.01m. Review before interpreting.')
old = 18.7651  # original central hybrid vs split-radial saving in €m/year
new = changes.loc[(changes.reference=='central_split_radial') & (changes.design=='central_hybrid'),'system'].iloc[0]
print(f'Original and independent endpoint savings: {old:.4f} and {new:.4f} million EUR/year')
previous = {"DK": 4.78, "DE": -9.23, "developer": 9.99}
endpoint = changes.query("reference == 'central_split_radial' and design == 'central_hybrid' and strike == 75 and pooled_cost_share == 1").iloc[0]
for account, original in previous.items():
    print(f"{account}: original {original:+.2f}, independent {endpoint[account]:+.2f} million EUR/year")